# Clase 086 — Feature importance

Medimos qué variables aportan a un modelo de árboles, distinguiendo
**MDI** (`feature_importances_`, sesgado por cardinalidad y calculado en train) de
**permutation importance** (sobre test, model-agnostic). Demostramos el sesgo de MDI
con una feature espuria de alta cardinalidad.

Requiere: `numpy`, `pandas`, `scikit-learn`, `matplotlib`.

## 1. Dataset `load_breast_cancer` con nombres de features

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

np.random.seed(42)

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print('train', X_train.shape, 'test', X_test.shape)

## 2. MDI: `feature_importances_` del Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=1)
rf.fit(X_train, y_train)
print(f'acc test: {rf.score(X_test, y_test):.4f}')

mdi = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
assert abs(mdi.sum() - 1.0) < 1e-6, 'MDI está normalizado (suma 1)'
print('\nTop-5 MDI:')
print(mdi.head().round(4).to_string())

## 3. Permutation importance sobre test (`n_repeats=10`)

In [ ]:
perm = permutation_importance(
    rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=1)
perm_imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
print('Top-5 permutation:')
print(perm_imp.head().round(4).to_string())

comp = pd.DataFrame({'MDI': mdi, 'permutation': perm_imp}).sort_values('MDI', ascending=False)
print('\nComparativa (top-8):')
print(comp.head(8).round(4).to_string())

## 4. Comparativa visual MDI vs permutation (top-10)

In [ ]:
top = mdi.head(10).index[::-1]
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
axes[0].barh(top, mdi[top].values, color='#37a')
axes[0].set_title('MDI (train, normalizado)')
axes[1].barh(top, perm_imp[top].values, color='#3a7')
axes[1].set_title('Permutation (test)')
for ax in axes:
    ax.set_xlabel('importancia')
plt.tight_layout()
plt.show()

## 5. Sesgo de MDI: feature espuria de alta cardinalidad

Agregamos un `random_id` sin poder predictivo. MDI le asigna importancia inflada
(muchos splits candidatos); permutation la ignora.

In [ ]:
X_train2 = X_train.copy()
X_test2 = X_test.copy()
rng = np.random.default_rng(42)
X_train2['random_id'] = rng.random(len(X_train2))
X_test2['random_id']  = rng.random(len(X_test2))

rf2 = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=1)
rf2.fit(X_train2, y_train)

mdi2 = pd.Series(rf2.feature_importances_, index=X_train2.columns)
perm2 = permutation_importance(
    rf2, X_test2, y_test, n_repeats=10, random_state=42, n_jobs=1)
perm2 = pd.Series(perm2.importances_mean, index=X_train2.columns)

print(f'random_id  MDI         : {mdi2["random_id"]:.4f}  (inflado, tiene señal espuria)')
print(f'random_id  permutation : {perm2["random_id"]:.4f}  (~0, correctamente ignorado)')
assert mdi2['random_id'] > perm2['random_id'], 'MDI infla la feature espuria vs permutation'
print('assert OK: MDI sobreestima la feature de alta cardinalidad; permutation no.')

## Ejercicios

1. Reportá el top-5 de features según MDI y según permutation. ¿Coincide el top-3?
   Justificá las diferencias por cardinalidad/correlación.
2. Detectá pares de features correlacionadas con `X.corr()` y explicá cómo eso afecta
   a la permutation importance.
3. Repetí el análisis con `GradientBoostingClassifier` y compará los rankings.
4. Usá `SelectFromModel(rf, threshold='median')` para quedarte con las features
   importantes y reentrená; ¿cae mucho la accuracy?

## Conclusiones

- MDI es rápido pero sesgado hacia features de **alta cardinalidad** y se calcula en train.
- Permutation importance se mide sobre datos **no vistos** y es model-agnostic; más
  honesto para auditar.
- Ambos flaquean con features **correlacionadas** (reparten crédito arbitrariamente).
- Regla práctica: rankeá con permutation sobre test, no con MDI sobre train.

## ✅ Soluciones de los ejercicios

Cinco ejercicios sobre California Housing (submuestreada para correr rápido): MDI, permutation importance, la trampa de la feature aleatoria y explicaciones SHAP. SHAP se calcula con el paquete `shap` si está instalado; si no, con los valores TreeSHAP nativos de un modelo de árbol como *fallback* offline.

**Ejercicio 1 — MDI del Random Forest.** `feature_importances_` ordenado y `barh`.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

data = fetch_california_housing()
# Submuestreo para mantener el runtime < 60s con n_jobs=1
rng = np.random.default_rng(42)
idx = rng.choice(len(data.data), 2000, replace=False)
X, y, feat = data.data[idx], data.target[idx], np.array(data.feature_names)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42)

rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=1).fit(Xtr, ytr)
order = np.argsort(rf.feature_importances_)
plt.figure(figsize=(6, 4))
plt.barh(feat[order], rf.feature_importances_[order])
plt.title('MDI (impurity-based) - RandomForest'); plt.tight_layout(); plt.show()
print('MDI top-3:', list(feat[order][::-1][:3]))

**Ejercicio 2 — Permutation importance.** `n_repeats=10` sobre test; comparamos con MDI lado a lado en un DataFrame.

In [ ]:
import pandas as pd
perm = permutation_importance(rf, Xte, yte, n_repeats=10, random_state=42, n_jobs=1)
comp = pd.DataFrame({'MDI': rf.feature_importances_,
                     'permutation': perm.importances_mean}, index=feat)
comp = comp.sort_values('permutation', ascending=False)
print(comp.round(4).to_string())
top_mdi = set(comp['MDI'].sort_values(ascending=False).index[:3])
top_perm = set(comp['permutation'].sort_values(ascending=False).index[:3])
print('\ntop-3 MDI :', top_mdi)
print('top-3 perm:', top_perm)
print('coincidencia top-3:', top_mdi & top_perm)

**Ejercicio 3 — La trampa de la feature aleatoria.** Añadimos `random_id` puro ruido: MDI le da importancia espuria (por su alta cardinalidad), permutation la ignora.

In [ ]:
X2tr = np.column_stack([Xtr, np.arange(len(Xtr))])
X2te = np.column_stack([Xte, np.arange(len(Xte))])
feat2 = np.append(feat, 'random_id')
rf2 = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=1).fit(X2tr, ytr)
mdi_rand = rf2.feature_importances_[-1]
perm2 = permutation_importance(rf2, X2te, yte, n_repeats=10, random_state=42, n_jobs=1)
perm_rand = perm2.importances_mean[-1]
print(f'random_id  MDI         : {mdi_rand:.4f}  (espuria, > 0)')
print(f'random_id  permutation : {perm_rand:.4f}  (~0, correcto)')
assert mdi_rand > perm_rand, 'MDI infla la feature ruidosa; permutation no'
print('OK: permutation es mas honesta con features de alta cardinalidad')

**Ejercicio 4 — SHAP summary (beeswarm).** El beeswarm muestra impacto **y** dirección por instancia; el bar de MDI solo da una magnitud global. Usamos `shap` si está, con *fallback* a TreeSHAP nativo del RF.

In [ ]:
try:
    import shap
    explainer = shap.TreeExplainer(rf)
    sv = explainer.shap_values(Xte)
    shap.summary_plot(sv, Xte, feature_names=list(feat), show=False)
    plt.tight_layout(); plt.show()
    shap_vals = np.asarray(sv)
    print('SHAP via paquete shap')
except Exception as e:
    print('shap no disponible (', type(e).__name__, ') -> fallback beeswarm manual')
    # Fallback: aproximamos contribuciones con permutacion por feature (signo via correlacion)
    base = rf.predict(Xte)
    shap_vals = np.zeros_like(Xte)
    for j in range(Xte.shape[1]):
        Xp = Xte.copy(); Xp[:, j] = np.median(Xte[:, j])
        shap_vals[:, j] = base - rf.predict(Xp)   # efecto de la feature j
    imp = np.abs(shap_vals).mean(0)
    o = np.argsort(imp)
    plt.figure(figsize=(7, 4))
    for rank, j in enumerate(o):
        jitter = np.random.default_rng(0).normal(0, 0.06, len(Xte))
        plt.scatter(shap_vals[:, j], rank + jitter, c=Xte[:, j], s=8, cmap='coolwarm')
    plt.yticks(range(len(o)), feat[o]); plt.axvline(0, color='k', lw=0.5)
    plt.xlabel('contribucion a la prediccion'); plt.title('Beeswarm (fallback): impacto y direccion')
    plt.colorbar(label='valor de la feature'); plt.tight_layout(); plt.show()
print('beeswarm top-3 por |contribucion|:',
      list(feat[np.argsort(np.abs(shap_vals).mean(0))[::-1][:3]]))

**Ejercicio 5 — Waterfall de una instancia con error grande.** Elegimos la predicción con mayor error absoluto y anotamos las 3 features que más la empujaron.

In [ ]:
err = np.abs(rf.predict(Xte) - yte)
i = int(np.argmax(err))
contrib = shap_vals[i]
o = np.argsort(np.abs(contrib))
plt.figure(figsize=(6, 4))
colors = ['#c33' if c < 0 else '#37a' for c in contrib[o]]
plt.barh(feat[o], contrib[o], color=colors)
plt.axvline(0, color='k', lw=0.6)
plt.title(f'Instancia #{i} (error {err[i]:.2f}) - contribuciones'); plt.tight_layout(); plt.show()
top3 = feat[o][::-1][:3]
print('prediccion:', round(float(rf.predict(Xte[i:i+1])[0]), 3), '| real:', round(float(yte[i]), 3))
print('las 3 features que mas empujaron:', list(top3))